In [ ]:
import os
import csv
import time
import random  
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
# Configuration parameters
class Config:
    model_path = "model/chinese_roberta_wwm_ext_pytorch"
    train_path = "train.csv"
    val_path = "val.csv"
    test_path = "test.csv"
    result_dir = "result_base"
    
    
    depth_dim = 64  
    thickness_dim = 64  
    lstm_hidden = 128
    dropout = 0.3
    
    batch_size = 32
    lr_bert = 3e-05
    lr_other = 1e-3
    weight_decay = 0.01
    epochs = 40
    patience = 5
    
    litho_weight = 0.7
    strata_weight = 0.3
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    def __init__(self):
        self.time_str = time.strftime("%m.%d_%H_%M")
        param_str = (f"{self.time_str}+"
                    f"dep{self.depth_dim}+"
                    f"thi{self.thickness_dim}+"
                    f"litho{self.litho_weight}+"
                    f"strata{self.strata_weight}")
        
        self.save_dir = os.path.join(self.result_dir, param_str)
        os.makedirs(self.save_dir, exist_ok=True)
        print(f"Results will be saved to: {self.save_dir}")

In [ ]:
# Custom dataset
class GeologyDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_len=160):
        self.data = []
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row['text']:  
                    self.data.append({
                        'text': row['text'],
                        'depth': float(row['depth']),
                        'thickness': float(row['thickness']),
                        'lithology': int(row['lithology_label']),
                        'strata': int(row['lithostratigraphic_unit_label'])
                    })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        encoding = self.tokenizer.encode_plus(
            item['text'],
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'depth': torch.tensor(item['depth'], dtype=torch.float),
            'thickness': torch.tensor(item['thickness'], dtype=torch.float),
            'lithology': torch.tensor(item['lithology'], dtype=torch.long),
            'strata': torch.tensor(item['strata'], dtype=torch.long)
        }

In [ ]:
# Gated attention unit
class EnhancedGAU(nn.Module):
    def __init__(self, input_dim):
        super(EnhancedGAU, self).__init__()
        # Gating mechanism
        self.gate_linear = nn.Sequential(
            nn.Linear(input_dim, input_dim * 2),
            nn.GLU(dim=-1)  
        )
        # Value transformation
        self.value_linear = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU()
        )
        # Layer normalization
        self.layer_norm = nn.LayerNorm(input_dim)
    
    def forward(self, features):
        # Gating mechanism
        gate_output = self.gate_linear(features)
        # Value transformation
        value_output = self.value_linear(features)
        # Gated fusion
        fused_features = gate_output * value_output
        # Residual connection and layer normalization
        output = self.layer_norm(features + fused_features)
        return output

In [ ]:
# Main model
class MultiTaskGeologyModel(nn.Module):
    def __init__(self, config):
        super(MultiTaskGeologyModel, self).__init__()
        self.config = config
        
        # Text encoder
        self.bert = BertModel.from_pretrained(config.model_path)
        self.bilstm = nn.LSTM(
            input_size=768,
            hidden_size=config.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.text_fc = nn.Linear(2 * config.lstm_hidden, 768)
        
        
        # Deep feature processing
        depth_mid = max(16, config.depth_dim // 2)
        self.depth_fc = nn.Sequential(
            nn.Linear(1, depth_mid),
            nn.ReLU(),
            nn.Linear(depth_mid, config.depth_dim)
        )
        
        # Thickness feature processing
        thickness_mid = max(16, config.thickness_dim // 2)
        self.thickness_fc = nn.Sequential(
            nn.Linear(1, thickness_mid),
            nn.ReLU(),
            nn.Linear(thickness_mid, config.thickness_dim)
        )
        
        # Feature fusion
        self.fusion_dim = 768 + config.depth_dim + config.thickness_dim
        self.gau = EnhancedGAU(self.fusion_dim)
        self.dropout = nn.Dropout(config.dropout)
        
        # Classification heads
        self.lithology_classifier = nn.Linear(self.fusion_dim, 7)
        self.strata_classifier = nn.Linear(self.fusion_dim, 8)
    
    def forward(self, input_ids, attention_mask, depth, thickness):
        # Text feature extraction
        bert_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sequence_output = bert_output.last_hidden_state
        
        # BiLSTM processing
        lstm_output, _ = self.bilstm(sequence_output)
        mask = attention_mask.unsqueeze(-1)
        lstm_output = lstm_output * mask
        lengths = mask.sum(dim=1)
        pooled_output = lstm_output.sum(dim=1) / lengths.clamp(min=1e-9)
        text_features = self.text_fc(pooled_output)
        
        # Numerical feature processing
        depth_features = self.depth_fc(depth.unsqueeze(1))
        thickness_features = self.thickness_fc(thickness.unsqueeze(1))
        
        # Feature fusion
        combined = torch.cat([text_features, depth_features, thickness_features], dim=1)
        fused_features = self.gau(combined)
        fused_features = self.dropout(fused_features)
        
        # Classification outputs
        litho_logits = self.lithology_classifier(fused_features)
        strata_logits = self.strata_classifier(fused_features)
        
        return litho_logits, strata_logits

In [ ]:
# Save results
def save_results(config, results, epoch, best=False):
    suffix = "_best" if best else f"_epoch{epoch}"
    filename = os.path.join(config.save_dir, f"results{suffix}.txt")
    
    with open(filename, 'w') as f:
        f.write(f"Training Configuration:\n")
        f.write(f"Lithology Weight: {config.litho_weight}, Strata Weight: {config.strata_weight}\n")
        f.write(f"Depth Dimension: {config.depth_dim}, Thickness Dimension: {config.thickness_dim}\n")
        f.write(f"Training Time: {config.time_str}\n\n")
        
        f.write("Lithology Classification Results:\n")
        f.write(f"Macro Precision: {results['lithology']['precision']:.4f}\n")
        f.write(f"Macro Recall: {results['lithology']['recall']:.4f}\n")
        f.write(f"Macro F1: {results['lithology']['f1']:.4f}\n")
        f.write("\nClassification Report:\n")
        f.write(results['lithology']['class_report'])
        f.write("\n" + "="*80 + "\n\n")
        
        f.write("Strata Classification Results:\n")
        f.write(f"Macro Precision: {results['strata']['precision']:.4f}\n")
        f.write(f"Macro Recall: {results['strata']['recall']:.4f}\n")
        f.write(f"Macro F1: {results['strata']['f1']:.4f}\n")
        f.write("\nClassification Report:\n")
        f.write(results['strata']['class_report'])
        f.write("\n" + "="*80 + "\n\n")
        
        f.write(f"Overall Macro F1: {results['macro_f1']:.4f}\n")
        f.write(f"Weighted F1: {results['weighted_f1']:.4f}\n")

In [ ]:
# Evaluation function
def evaluate(model, dataloader, config, task='all'):
    model.eval()
    litho_preds, litho_targets = [], []
    strata_preds, strata_targets = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            depth = batch['depth'].to(config.device)
            thickness = batch['thickness'].to(config.device)
            lithology = batch['lithology'].to(config.device)
            strata = batch['strata'].to(config.device)
            
            litho_logits, strata_logits = model(
                input_ids, attention_mask, depth, thickness
            )
            
            litho_preds.extend(torch.argmax(litho_logits, dim=1).cpu().numpy())
            litho_targets.extend(lithology.cpu().numpy())
            
            strata_preds.extend(torch.argmax(strata_logits, dim=1).cpu().numpy())
            strata_targets.extend(strata.cpu().numpy())
    
    results = {}
    
    # Lithology classification evaluation
    litho_precision = precision_score(litho_targets, litho_preds, average='macro', zero_division=0)
    litho_recall = recall_score(litho_targets, litho_preds, average='macro', zero_division=0)
    litho_f1 = f1_score(litho_targets, litho_preds, average='macro', zero_division=0)
    
    # Generate report using sklearn's classification_report
    litho_report = classification_report(
        litho_targets, litho_preds, 
        target_names=[f'Class_{i}' for i in range(7)],
        digits=4,
        zero_division=0
    )
    
    results['lithology'] = {
        'precision': litho_precision,
        'recall': litho_recall,
        'f1': litho_f1,
        'class_report': litho_report
    }
    
    # Strata classification evaluation
    strata_precision = precision_score(strata_targets, strata_preds, average='macro', zero_division=0)
    strata_recall = recall_score(strata_targets, strata_preds, average='macro', zero_division=0)
    strata_f1 = f1_score(strata_targets, strata_preds, average='macro', zero_division=0)
    
    strata_report = classification_report(
        strata_targets, strata_preds, 
        target_names=[f'Class_{i}' for i in range(8)],
        digits=4,
        zero_division=0
    )
    
    results['strata'] = {
        'precision': strata_precision,
        'recall': strata_recall,
        'f1': strata_f1,
        'class_report': strata_report
    }
    
    # Comprehensive evaluation
    results['macro_f1'] = (litho_f1 + strata_f1) / 2
    results['weighted_f1'] = (config.litho_weight * litho_f1 + 
                              config.strata_weight * strata_f1)
    
    return results

In [ ]:
# Main training function
def train_and_evaluate(config):
    # Initialization
    tokenizer = BertTokenizer.from_pretrained(config.model_path)
    
    # Create datasets
    train_dataset = GeologyDataset(config.train_path, tokenizer)
    val_dataset = GeologyDataset(config.val_path, tokenizer)
    test_dataset = GeologyDataset(config.test_path, tokenizer)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size)
    
    # Initialize the model
    model = MultiTaskGeologyModel(config).to(config.device)
    
    # Loss functions
    litho_criterion = nn.CrossEntropyLoss()
    strata_criterion = nn.CrossEntropyLoss()
    
    # Optimizer
    bert_params = list(model.bert.parameters())
    other_params = list(model.bilstm.parameters()) + \
                   list(model.text_fc.parameters()) + \
                   list(model.depth_fc.parameters()) + \
                   list(model.thickness_fc.parameters()) + \
                   list(model.gau.parameters()) + \
                   list(model.lithology_classifier.parameters()) + \
                   list(model.strata_classifier.parameters())
    
    optimizer = optim.AdamW([
        {'params': bert_params, 'lr': config.lr_bert},
        {'params': other_params, 'lr': config.lr_other}
    ], weight_decay=config.weight_decay)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',         
        factor=0.5,        
        patience=2,         
        min_lr=1e-6       
    )
    
    # Training loop
    best_val_f1 = 0
    patience_counter = 0
    history = []
    
    for epoch in range(config.epochs):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            # Prepare data
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            depth = batch['depth'].to(config.device)
            thickness = batch['thickness'].to(config.device)
            lithology = batch['lithology'].to(config.device)
            strata = batch['strata'].to(config.device)
            
            # Forward pass
            litho_logits, strata_logits = model(
                input_ids, attention_mask, depth, thickness
            )
            
            # Calculate multi-task loss
            litho_loss = litho_criterion(litho_logits, lithology)
            strata_loss = strata_criterion(strata_logits, strata)
            loss = config.litho_weight * litho_loss + config.strata_weight * strata_loss
            
            # Backward pass
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            total_loss += loss.item()
        
        # Validation evaluation
        val_results = evaluate(model, val_loader, config)
        

        val_f1 = val_results['macro_f1'] 
        
        # Update learning rate
        scheduler.step(val_f1)
        
        # Log learning rate changes
        if epoch == 0 or scheduler.last_epoch % scheduler.patience == 0:
            lrs = [group['lr'] for group in optimizer.param_groups]
            print(f"Current Learning Rate: BERT group={lrs[0]:.2e}, Other layers={lrs[1]:.2e}")
        
        # Save training history
        history.append({
            'epoch': epoch + 1,
            'train_loss': total_loss / len(train_loader),
            'val_litho_f1': val_results['lithology']['f1'],
            'val_strata_f1': val_results['strata']['f1'],
            'val_composite_f1': val_results['macro_f1']  
        })
        
        # Log training progress
        print(f"Epoch {epoch+1}/{config.epochs} | "
              f"Train Loss: {total_loss/len(train_loader):.4f} | "
              f"Val Litho F1: {val_results['lithology']['f1']:.4f} | "
              f"Val Strata F1: {val_results['strata']['f1']:.4f} | "
              f"Val Composite F1: {val_results['macro_f1']:.4f}")  
        
        # Save the best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(config.save_dir, "best_model.pth"))
            save_results(config, val_results, epoch, best=True)
            print(f"Saving new best model, Composite F1: {val_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                print(f"Early stopping triggered (no improvement in Composite F1). Training stopped at epoch {epoch+1}.")
                break
    
    # Final evaluation on test set
    model.load_state_dict(torch.load(os.path.join(config.save_dir, "best_model.pth")))
    test_results = evaluate(model, test_loader, config)
    
    # Save final results
    save_results(config, test_results, "final")
    
    # Save training history
    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(config.save_dir, "training_history.csv"), index=False)
    
    # Print final performance metrics
    print("\nFinal Test Results:")
    print(f"Lithology Macro F1: {test_results['lithology']['f1']:.4f}")
    print(f"Strata Macro F1: {test_results['strata']['f1']:.4f}")
    print(f"Composite Macro F1: {test_results['macro_f1']:.4f}")
    
    return test_results

In [ ]:
# Main execution
if __name__ == "__main__":
    # Set random seed for reproducibility
    set_seed(42)
    print(f"Random seed fixed: 42")
    
    config = Config()
    print(f"Using device: {config.device}")
    print(f"Lithology Weight: {config.litho_weight}, Strata Weight: {config.strata_weight}")
    print(f"Depth Dimension: {config.depth_dim}, Thickness Dimension: {config.thickness_dim}")
    
    start_time = time.time()
    results = train_and_evaluate(config)
    end_time = time.time()
    
    print(f"\nTotal elapsed time: {(end_time - start_time)/60:.2f} minutes")
    print(f"Results saved to: {config.save_dir}")